In [ ]:
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct # These are the different types of distance metrics that can be used for vector search, and the structure of a point in the vector database.
# Distance: This is a class that defines the different types of distance metrics that can be used for vector search, such as cosine similarity, Euclidean distance, etc, for the VECTOR DATABASE.
# VectorParams: This is a class that defines the parameters for the vector database, such as embedding size and distance metric , for the VECTOR DATABASE.
# PointStruct: This is a class that defines the structure of a point in the vector database, which includes an id, a vector, and optional payload data; i.e. metadata, 

import openai

### Read the sampled dataset with Amazon inventory data

In [4]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)    


In [5]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,"MUQI Bluetooth Speaker, Portable Bluetooth Spe...",4.4,113,[IPX7 Waterproof Bluetooth Speaker: The silico...,[],15.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Powerful sound, compact size, and ...",MUQI,"[Electronics, Portable Audio & Video, Portable...","{'Product Dimensions': '6.9 x 2 x 2.6 inches',...",B0B3WPPBZ8,NaN,NaN,NaN
1,All Electronics,DEEBOX Wireless Gaming Headsets with Mic for P...,4.6,642,[✅【Focused Voice Pickup】In-built mic cancels b...,[],59.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Wireless Gaming Headset', 'url': '...",DEEBOX,[],{'Package Dimensions': '8.11 x 6.97 x 4.57 inc...,B0C8J2CCKV,NaN,NaN,NaN
2,Computers,"USB Flash Drive 1TB, Portable Thumb Drive 1000...",4.1,229,[Ultra Large Capacity USB Drive - USB Flash Dr...,[Manage and organize all your most important d...,19.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Amazing Value USB Memory Stick Va...,NOOYU,"[Electronics, Computers & Accessories, Data St...","{'Brand': 'NOOYU', 'Item Weight': '0.563 ounce...",B0C6FDCK21,NaN,NaN,NaN
3,Camera & Photo,"REOLINK PTZ Camera Outdoor, 5MP HD WiFi Camera...",3.9,128,[5MP HD & 3X OPTICAL ZOOM: With the resolution...,[],109.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Reolink Trackmix WIFI and PoE Revi...,REOLINK,"[Electronics, Camera & Photo, Video Surveillan...",{'Product Dimensions': '4.65 x 3.35 x 3.35 inc...,B0B61FHY76,NaN,NaN,NaN
4,Camera & Photo,Merkury Innovations Indoor Smart Security Came...,4.2,416,[SMART SECURITY: The Merkury Innovations Smart...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Much easier to set up than the oth...,Merkury Innovations,"[Electronics, Camera & Photo, Video Surveillan...",{'Product Dimensions': '3.54 x 3.45 x 5.91 inc...,B0B4RBPQKB,NaN,NaN,NaN


In [6]:
list(df_items["features"].items())[0]

(0,
 ['IPX7 Waterproof Bluetooth Speaker: The silicone case and closed port cover of this waterproof bluetooth speaker provide strong waterproof & dustproof protection, even submerged in water; can be used in bathroom, beach, swimming pool without worrying about rain or spills, it is an excellent shower speaker, hiking speaker, camping speaker,car speaker.',
  'Clear and Loud Stereo: This wireless stereo speaker has 2 built-in drivers to provide 10w audio power for loud and clear sound; TWS technology allows you to pair two portable bluetooth speakers with your phone, dual pairing to enjoy true wireless stereo.If you want to rock, get this! You will be happy!',
  'Long battery life: This portable bluetooth speaker is equipped with a 2200mAh Li-ion battery, which only takes 3 hours to charge and can play for 24 hours at 50% volume, making it easy to use and play all day.MUQI wireless Bluetooth portable small speakers, carnival party to add atmosphere essential items, high-quality sound 

In [7]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51Kn6oexQkL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51Kn6oexQkL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/81gDfqCHSeL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/512U86nKD8L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/512U86nKD8L._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71YLvhZhGLL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41QcjmISEJL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41QcjmISEJL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/61LVJjp6VrL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41BT1cSQw5L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41BT1cSQw5L._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71b2+xvdftL._AC_SL1

### Preprocessing title and features

In [8]:
# New string - Description combining title and features

def preprocess_description(row):
    return f"{row['title']}{' '.join(row['features'])}"

In [9]:
def extract_first_large_image(row):
    return row['images'][0].get("large", "") # Extract the 'large' image URL from the first image dictionary if available, else return an empty string

In [11]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)

In [12]:
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [13]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,All Electronics,"MUQI Bluetooth Speaker, Portable Bluetooth Spe...",4.4,113,[IPX7 Waterproof Bluetooth Speaker: The silico...,"MUQI Bluetooth Speaker, Portable Bluetooth Spe...",15.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Powerful sound, compact size, and ...",MUQI,"[Electronics, Portable Audio & Video, Portable...","{'Product Dimensions': '6.9 x 2 x 2.6 inches',...",B0B3WPPBZ8,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51Kn6oexQk...
1,All Electronics,DEEBOX Wireless Gaming Headsets with Mic for P...,4.6,642,[✅【Focused Voice Pickup】In-built mic cancels b...,DEEBOX Wireless Gaming Headsets with Mic for P...,59.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Wireless Gaming Headset', 'url': '...",DEEBOX,[],{'Package Dimensions': '8.11 x 6.97 x 4.57 inc...,B0C8J2CCKV,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41we2iBE2P...
2,Computers,"USB Flash Drive 1TB, Portable Thumb Drive 1000...",4.1,229,[Ultra Large Capacity USB Drive - USB Flash Dr...,"USB Flash Drive 1TB, Portable Thumb Drive 1000...",19.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Amazing Value USB Memory Stick Va...,NOOYU,"[Electronics, Computers & Accessories, Data St...","{'Brand': 'NOOYU', 'Item Weight': '0.563 ounce...",B0C6FDCK21,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31mOkS1N93...
3,Camera & Photo,"REOLINK PTZ Camera Outdoor, 5MP HD WiFi Camera...",3.9,128,[5MP HD & 3X OPTICAL ZOOM: With the resolution...,"REOLINK PTZ Camera Outdoor, 5MP HD WiFi Camera...",109.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Reolink Trackmix WIFI and PoE Revi...,REOLINK,"[Electronics, Camera & Photo, Video Surveillan...",{'Product Dimensions': '4.65 x 3.35 x 3.35 inc...,B0B61FHY76,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31m6CwBLCQ...
4,Camera & Photo,Merkury Innovations Indoor Smart Security Came...,4.2,416,[SMART SECURITY: The Merkury Innovations Smart...,Merkury Innovations Indoor Smart Security Came...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Much easier to set up than the oth...,Merkury Innovations,"[Electronics, Camera & Photo, Video Surveillan...",{'Product Dimensions': '3.54 x 3.45 x 5.91 inc...,B0B4RBPQKB,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41mFtR5t1W...


### Sample 50 Itemd from the dataset

In [14]:
df_sample = df_items.sample(50, random_state=42)
len(df_sample)

50

In [15]:
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")


In [16]:
data_to_embed[0:2]

# Here we have a list of dictionaries which can each be indiv processed for embedding and storage in Qdrant; our vector DB

[{'description': 'COOLHOOD 15.6 Inch Portable Monitor, Ultra Slim&Light Portable Laptop Monitor (15.6")',
  'image': 'https://m.media-amazon.com/images/I/51uWC9HIORL._AC_.jpg',
  'rating_number': 111,
  'price': nan,
  'average_rating': 4.6,
  'parent_asin': 'B0BGMY79D5'},
 {'description': 'Google Nest Cam Outdoor or Indoor, Battery - 2nd Generation - 1 PackNest Cam has built-in intelligence and can tell the difference between a person, animal, and vehicle and send alerts directly through the Google Home app[1], no subscription required.Controller Type:Google Assistant.Connectivity protocol:Bluetooth;Wi-Fi.Power source type:Battery Powered Easily check in from anywhere 24/7 with 1080p HDR video[1] with night vision, and see what you missed with 3 hours of free event video history[2]; add a Nest Aware subscription (sold separately) for up to 60 days of video history[3] If your Wi-Fi goes down or there’s a power outage, Nest Cam will store up to an hour of recorded events so you can see 

### Embedding our data - Defining our embedding function

In [17]:
# OpenAI embedding example -  
# this embedding algorithm ; A deep transformer model trained to minimise semantic distance between related texts and maximise distance between unrelated ones.

response = openai.embeddings.create( 
    input="This is a sample description for embedding.",
    model="text-embedding-3-small"
)   

response.data[0].embedding

[-0.00025012617697939277,
 0.02688148431479931,
 0.022186147049069405,
 0.0020576417446136475,
 0.0012210275745019317,
 -0.038990508764982224,
 0.02740318700671196,
 -0.017531998455524445,
 -0.00655219005420804,
 0.028556426987051964,
 0.01587078347802162,
 -0.012953360565006733,
 0.022350896149873734,
 -0.04437229409813881,
 0.0023699775338172913,
 0.011216634884476662,
 -0.005395517218858004,
 0.043136682361364365,
 0.0036519276909530163,
 0.06859034299850464,
 0.021238842979073524,
 -0.027609122917056084,
 -0.019412878900766373,
 0.06507570296525955,
 0.005525943357497454,
 -0.04818897321820259,
 -0.004527154844254255,
 0.03783726692199707,
 0.07501553744077682,
 -0.0011214918922632933,
 0.023119723424315453,
 -0.04091257601976395,
 0.011834442615509033,
 -0.024163130670785904,
 -0.007550978567451239,
 0.022460728883743286,
 0.04643165320158005,
 0.006600241642445326,
 -0.0025295778177678585,
 -0.0022120934445410967,
 0.016227738931775093,
 -0.042862098664045334,
 -0.004310922231525

In [18]:
len(response.data[0].embedding)

1536

In [19]:
# Function : Return embedding for a given text using OpenAI's embedding API. 
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [20]:
get_embedding("This is another sample description.")

[-0.001875470974482596,
 0.015956874936819077,
 -0.024965284392237663,
 0.00015048444038257003,
 0.006740934681147337,
 -0.014465722255408764,
 -0.006218262482434511,
 0.0013989168219268322,
 0.01571091264486313,
 0.006848543416708708,
 -0.019692445173859596,
 -0.03600289300084114,
 0.010945371352136135,
 -0.04378148540854454,
 0.03818581998348236,
 0.04015352576971054,
 -0.01872396469116211,
 0.03812432661652565,
 -0.028962191194295883,
 0.05678680166602135,
 0.03750941902399063,
 -0.0029842278454452753,
 0.019738562405109406,
 0.044580865651369095,
 0.012321229092776775,
 -0.04501130431890488,
 -0.025918394327163696,
 -0.007932319305837154,
 0.06776291877031326,
 -0.03240567818284035,
 0.028085945174098015,
 -0.04867000877857208,
 0.005937710404396057,
 -0.02899293601512909,
 0.010568739846348763,
 0.023673977702856064,
 0.04983833432197571,
 -0.01068403571844101,
 -0.005603353958576918,
 -0.02890069968998432,
 -0.001173130120150745,
 -0.06253619492053986,
 0.013051432557404041,
 0.0

### Create a Qdrant collection


In [21]:
qdrant_client = QdrantClient(url="http://localhost:6333") # Connect to the Qdrant vector database running in the Docker container

In [22]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-00", # Name of the collection in Qdrant where our item embeddings will be stored
    vectors_config= VectorParams(size= 1536, distance=Distance.COSINE) 
)

True

#### Embed data

In [ ]:
# Test 1

# PointStruct is a data structure used to represent a point in the vector space, which includes;
# 1. ID 
# 2. the vector itself (the embedding), 
# 3. additional payload (metadata) associated with that point

pointstruct = PointStruct(
    id=0,
    vector = get_embedding("Test text"),
    payload = {
        "test": "Test text",
        "model": "text-embedding-3-small"
    }
)


In [25]:
pointstruct

PointStruct(id=0, vector=[-0.020057253539562225, 0.006970119196921587, 0.037700485438108444, -0.040323127061128616, -0.01916317082941532, -0.0343029722571373, 0.0005480913096107543, -0.02425944246351719, 0.03871377930045128, 0.0015953787369653583, 0.030667034909129143, 0.012792832218110561, -0.010863103903830051, 0.011287793517112732, 0.02639034017920494, 0.04303517937660217, -0.046820126473903656, -0.00690678833052516, -0.02174110896885395, 0.05108192190527916, 0.0070856050588190556, 0.013113211840391159, 0.01130269467830658, -0.032544609159231186, -0.00393396383151412, -0.039041608572006226, -0.036955416202545166, 0.004965884145349264, 0.05841339752078056, -0.07462609559297562, 0.031382299959659576, -0.044018667191267014, -0.00025891143013723195, -0.010974864475429058, 0.006005255039781332, 0.03814752772450447, 0.03030940145254135, 0.037640880793333054, 0.010274499654769897, -0.029355714097619057, -0.014916278421878815, -0.010386260226368904, 0.021279167383909225, 0.01814987696707248

### Amazon Data


##### 1. Create list of the Embedded data 

In [ ]:
# The description field is part of the pay that will be embedded, 
# we use payload to store the other metadata fields that we want to be able to retrieve when we query the vector database.

pointstructs=[] # List of PointStruct objects to be stored in Qdrant; each PointStruct represents an embedded item with its metadata.

for i,data in enumerate(data_to_embed):
    embedding = get_embedding(data["description"]) # Embed description of each item
    pointstructs.append(
        PointStruct(
        id = i, 
        vector = embedding, 
        payload = data # Store the original data (description, image, rating_number, price, average_rating, parent_asin) as metadata
                    )
                       )

In [27]:
pointstructs

[PointStruct(id=0, vector=[0.026413757354021072, -0.01664472371339798, -0.042236387729644775, -0.0044387709349393845, 0.00407843803986907, -0.0122993728145957, -0.010180080309510231, 0.0013172179460525513, 0.044200871139764786, 0.002767892787232995, -0.03698353096842766, -0.010543081909418106, -0.05628671497106552, 0.05269939452409744, 0.04078437760472298, -0.0037394578102976084, -0.03852095082402229, 0.004468131344765425, -0.06610912829637527, 0.014947154559195042, 0.032264500856399536, 0.05385246127843857, 0.031922850757837296, 0.002129969419911504, 0.06730490177869797, -0.019997157156467438, 0.0003573303984012455, -0.02718246728181839, 0.019239123910665512, 0.0017162535805255175, 0.05111926794052124, -0.027054348960518837, 0.02190825715661049, -0.08558313548564911, -0.01193637028336525, 0.012897258624434471, -0.028335534036159515, -0.007137266453355551, -0.030897904187440872, 0.0055998447351157665, 0.02102210558950901, 0.02190825715661049, 0.0849425420165062, 0.03691947087645531, 0.

##### 2. Write to Qdrant

In [28]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True, # Wait for the operation to complete before proceeding; ensures that the points are fully indexed and available for search immediately after upsert.
    points=pointstructs, # List of PointStruct objects to be upserted into the Qdrant collection
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

##### 3. Retrieval function

In [32]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query) # Embed the query using the same embedding model
    result = qdrant_client.query_points(  # Query the Qdrant collection for the top k most similar points to the query embedding
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )
    return result

##### 4. Test Retrieval

In [33]:
retrieve_data("What kind of charging cords do you offer?", k=10).points # Returns top10 most similar items to the query based on cosine similarity of their embeddings as a list.

[ScoredPoint(id=14, version=1, score=0.43398994, payload={'description': "iPhone Charger Cord [Apple MFi Certified] Lightning Cable 1m/3.3FT Fast Charging High Speed Data Sync USB Cable Compatible with iPhone 14/13/12/11 Pro Max/XS MAX/XR/XS/X/8/7/Plus/6S iPad[Fast Charging & Sync] - iPhone Charger Cord Max 2.4 output current, high quality copper wire, improve charging and data transfer speed of iPhone charging cable. iPhone Lightning Cable ensures faster charging time and keeps your device safer. [Perfect comptibility] - iPhone lightning cables compatible with iPhone 14/14 Plus/14 Pro/14 Pro Max/iPhone 13/13 mini/13 Pro/13 Pro Max/12/12 mini/12 Pro/12 Pro Max/11/11 Pro/11 Pro Max/SE 2/XS/XS Max/XR/X/8Plus/8/7Plus/7/6s Plus/6s/6 Plus/6/5s/5c,iPad,AirPods [MFi Certified] - The lightning cable has an original lightning chip inside. Make sure it's compatible with your iPhone and charges your device safely. Enjoy fast data transfer, sync and charging. [Superior Durability] - iPhone lightni